In [1]:
from pathlib import Path
import csv
import json

artifact_dir = Path("artifacts/agent_rounds/20260615_round7/artifact_inventory_need")
artifact_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    "round": "20260615_round7",
    "scenario": "artifact_inventory_need",
    "baseline_accuracy": 0.781,
    "n_eval": 12,
}
(artifact_dir / "baseline_metrics.json").write_text(
    json.dumps(metrics, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

with (artifact_dir / "validation_predictions.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["row_id", "label", "prediction", "score"])
    writer.writerow([1, "ok", "ok", "0.91"])
    writer.writerow([2, "drift", "ok", "0.43"])
    writer.writerow([3, "ok", "ok", "0.86"])

(artifact_dir / "mutable_summary.txt").write_text(
    "seed summary: pending repair\n",
    encoding="utf-8",
)

print(f"artifact_dir={artifact_dir.as_posix()}")
print("created:")
print("- baseline_metrics.json")
print("- validation_predictions.csv")
print("- mutable_summary.txt")


artifact_dir=artifacts/agent_rounds/20260615_round7/artifact_inventory_need
created:
- baseline_metrics.json
- validation_predictions.csv
- mutable_summary.txt


In [2]:
from pathlib import Path
import json

artifact_dir = Path("artifacts/agent_rounds/20260615_round7/artifact_inventory_need")
artifact_dir.mkdir(parents=True, exist_ok=True)

(artifact_dir / "mutable_summary.txt").write_text(
    "repair summary: model threshold lowered after inspecting failed probe\n",
    encoding="utf-8",
)

repaired_nested = artifact_dir / "nested" / "repaired_cell" / "final_probe.txt"
repaired_nested.parent.mkdir(parents=True, exist_ok=True)
repaired_nested.write_text(
    "final probe from repaired cell\n",
    encoding="utf-8",
)

leftovers = [
    "failure_only.json",
    "nested/failed_cell/stale_probe.txt",
]
repair_summary = {
    "cell": "02_repair_artifacts",
    "status": "repaired",
    "leftover_failure_artifacts": {
        path: (artifact_dir / path).exists()
        for path in leftovers
    },
}
(artifact_dir / "repair_summary.json").write_text(
    json.dumps(repair_summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("repair wrote:")
print("- mutable_summary.txt")
print("- nested/repaired_cell/final_probe.txt")
print("- repair_summary.json")
for path in leftovers:
    print(f"leftover_exists {path}={(artifact_dir / path).exists()}")


repair wrote:
- mutable_summary.txt
- nested/repaired_cell/final_probe.txt
- repair_summary.json
leftover_exists failure_only.json=True
leftover_exists nested/failed_cell/stale_probe.txt=True


In [3]:
from pathlib import Path
import json

artifact_dir = Path("artifacts/agent_rounds/20260615_round7/artifact_inventory_need")
metrics = json.loads((artifact_dir / "baseline_metrics.json").read_text(encoding="utf-8"))
repair = json.loads((artifact_dir / "repair_summary.json").read_text(encoding="utf-8"))

decision = "hold" if metrics["baseline_accuracy"] < 0.80 else "ship"
report = [
    "# Downstream Decision",
    "",
    f"- baseline_accuracy: {metrics['baseline_accuracy']}",
    f"- decision: {decision}",
    f"- repair_status: {repair['status']}",
    f"- failure_only_leftover: {repair['leftover_failure_artifacts']['failure_only.json']}",
]
(artifact_dir / "decision_report.md").write_text("\n".join(report) + "\n", encoding="utf-8")

confusion = "label,prediction,count\nok,ok,10\ndrift,ok,2\n"
(artifact_dir / "confusion_matrix.csv").write_text(confusion, encoding="utf-8")

print(f"decision={decision}")
print("created:")
print("- decision_report.md")
print("- confusion_matrix.csv")


decision=hold
created:
- decision_report.md
- confusion_matrix.csv


In [4]:
from pathlib import Path
import json

artifact_dir = Path("artifacts/agent_rounds/20260615_round7/artifact_inventory_need")

pre_snapshot = [
    {
        "path": path.relative_to(artifact_dir).as_posix(),
        "bytes": path.stat().st_size,
    }
    for path in sorted(artifact_dir.rglob("*"))
    if path.is_file()
]

(artifact_dir / "inventory_snapshot.json").write_text(
    json.dumps(pre_snapshot, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

final_snapshot = [
    {
        "path": path.relative_to(artifact_dir).as_posix(),
        "bytes": path.stat().st_size,
    }
    for path in sorted(artifact_dir.rglob("*"))
    if path.is_file()
]

print("artifact inventory from notebook cell:")
for item in final_snapshot:
    print(f"- {item['path']} ({item['bytes']} bytes)")

print("classified leftovers:")
print(f"- failure_only.json stale_leftover={(artifact_dir / 'failure_only.json').exists()}")
print(
    "- nested/failed_cell/stale_probe.txt stale_leftover="
    f"{(artifact_dir / 'nested' / 'failed_cell' / 'stale_probe.txt').exists()}"
)
print(f"total_files={len(final_snapshot)}")


artifact inventory from notebook cell:
- baseline_metrics.json (120 bytes)
- confusion_matrix.csv (43 bytes)
- decision_report.md (123 bytes)
- failure_only.json (129 bytes)
- inventory_snapshot.json (651 bytes)
- mutable_summary.txt (70 bytes)
- nested/failed_cell/stale_probe.txt (81 bytes)
- nested/repaired_cell/final_probe.txt (31 bytes)
- repair_summary.json (177 bytes)
- validation_predictions.csv (76 bytes)
classified leftovers:
- failure_only.json stale_leftover=True
- nested/failed_cell/stale_probe.txt stale_leftover=True
total_files=10
